# WEARO AI Lab — FASHN VTON 1.5 Colab

**Purpose:** reproduce and evaluate the same FASHN VTON 1.5 inference used by WEARO's `gpu-tryon` service. This notebook is for R&D only; it is not the production API.

Production contract currently uses `POST /v1/try-on` with person/garment image URLs, while the GPU service loads `TryOnPipeline` from the FASHN VTON package.

In [ ]:
# Confirm Colab is using a GPU before installing the model stack.
!nvidia-smi


In [ ]:
# Clone the exact WEARO branch under test.
%cd /content
!rm -rf fashion-dream-try-on-shop
!git clone --branch feature/product-admin --depth 1 https://github.com/minhquanto82kt/fashion-dream-try-on-shop.git
%cd /content/fashion-dream-try-on-shop/fashion-dream-try-on-main/gpu-tryon
!git rev-parse HEAD

In [ ]:
# Install the same FASHN VTON package declared by the repository's GPU service.
!pip install -q -r requirements.txt
!pip install -q huggingface_hub


In [ ]:
# Download the same weights used by gpu-tryon, into the Colab runtime.
import os
os.environ['VTON_WEIGHTS_DIR'] = '/content/fashn-vton-models'
os.environ['VTON_OUTPUT_DIR'] = '/content/tryon-output'
!python scripts/download_weights.py

In [ ]:
# Upload one person image and one garment image. Do not upload customer-identifying images for experiments.
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))


## Input convention

Set the filenames below. The production GPU service accepts `tops`, `bottoms`, and `one-pieces`; use `tops` for the first baseline experiment.

In [ ]:
PERSON_IMAGE = '/content/person.jpg'      # change to uploaded filename
GARMENT_IMAGE = '/content/garment.jpg'    # change to uploaded filename
CATEGORY = 'tops'
GARMENT_PHOTO_TYPE = 'model'
NUM_TIMESTEPS = 30
GUIDANCE_SCALE = 1.5
SEED = 42


In [ ]:
from pathlib import Path
from PIL import Image
from fashn_vton import TryOnPipeline
import time

assert Path(PERSON_IMAGE).is_file(), f'Missing person image: {PERSON_IMAGE}'
assert Path(GARMENT_IMAGE).is_file(), f'Missing garment image: {GARMENT_IMAGE}'

person = Image.open(PERSON_IMAGE).convert('RGB')
garment = Image.open(GARMENT_IMAGE).convert('RGB')
print('Person:', person.size)
print('Garment:', garment.size)

pipeline = TryOnPipeline(weights_dir='/content/fashn-vton-models')
start = time.perf_counter()
result = pipeline(
    person_image=person,
    garment_image=garment,
    category=CATEGORY,
    garment_photo_type=GARMENT_PHOTO_TYPE,
    num_samples=1,
    num_timesteps=NUM_TIMESTEPS,
    guidance_scale=GUIDANCE_SCALE,
    seed=SEED,
    segmentation_free=True,
)
elapsed = time.perf_counter() - start
assert result.images, 'FASHN VTON returned no images'
output_path = '/content/tryon-output/tryon-result.png'
Path('/content/tryon-output').mkdir(parents=True, exist_ok=True)
result.images[0].save(output_path, format='PNG')
print(f'Inference time: {elapsed:.2f}s')
print('Output:', output_path)

In [ ]:
from IPython.display import display
display(Image.open(PERSON_IMAGE))
display(Image.open(GARMENT_IMAGE))
display(Image.open('/content/tryon-output/tryon-result.png'))


## Baseline evaluation

Record the result after each run instead of judging only by whether inference completed.

- Model: FASHN VTON 1.5
- Category: `tops` / `bottoms` / `one-pieces`
- Garment photo type: `model` / `flat-lay`
- Timesteps: 30
- Guidance scale: 1.5
- Seed: 42
- Inference time: record from the cell output
- Visual quality: evaluate garment fidelity, body/pose preservation, face/hair preservation, hands/edges, and background stability

Do not commit customer images or generated images to GitHub.

In [ ]:
# Optional: run a small parameter comparison with the same input pair.
# Keep the seed fixed so the comparison is easier to interpret.
EXPERIMENTS = [
    {'num_timesteps': 30, 'guidance_scale': 1.5},
    {'num_timesteps': 40, 'guidance_scale': 1.5},
    {'num_timesteps': 30, 'guidance_scale': 2.0},
]

print('Configured experiments:', EXPERIMENTS)
print('Run these one at a time when GPU memory/time allows, then record qualitative observations.')

## Promotion rule

A Colab experiment is **not** a production change. If a configuration/model change is validated, implement it in the appropriate repository service (`gpu-tryon` or `python`) and verify the existing production contract before deployment.

Production flow remains: `WEARO → Python FastAPI → gpu-tryon → FASHN VTON → Supabase Storage → signed URL`.